<a href="https://colab.research.google.com/github/Rebeca0310/Treinamento/blob/main/Exercicio_ATM_Revisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Programação Orientada a Objetos em Python
Neste projeto, você deverá desenvolver um sistema bancário simplificado utilizando os principais conceitos de Programação Orientada a Objetos (POO) em Python, como:

* Herança

* Abstração

* Encapsulamento

* Polimorfismo

In [ ]:
class Cliente:

    def __init__(self, nome: str, cpf: str, email: str):
        self.nome: str = nome
        self.cpf: str = cpf
        self.email: str = email
        self.contas: list[Conta] = []

    def adicionar_conta(self, conta: "Conta") -> None:
        if conta not in self.contas:
            self.contas.append(conta)

    def listar_contas(self) -> None:
        print(f"\n Contas de {self.nome} ---")
        if not self.contas:
            print("Nenhuma conta vinculada.")
            return
        for conta in self.contas:
            tipo_txt = (
                "Corrente" if isinstance(conta, ContaCorrente) else "Poupança"
            )
            print(
                f"- Conta {tipo_txt} Nº: {conta.numero} | Saldo: R$ {conta.consultar_saldo():.2f}"
            )

In [ ]:
from datetime import datetime
from enum import Enum

class TipoTransacao(Enum):
    DEPOSITO = "Depósito"
    SAQUE = "Saque"
    TRANSFERENCIA_ENVIADA = "Transferência Enviada"
    TRANSFERENCIA_RECEBIDA = "Transferência Recebida"

class Transacao:
#método construtor
    def __init__(self, tipo: TipoTransacao, valor: float):
       #atributos
        self.tipo: TipoTransacao = tipo
        self.valor: float = valor
        self.data_hora: datetime = datetime.now()
#método instancia
    def __str__(self) -> str:
        data_formatada = self.data_hora.strftime("%d/%m/%Y %H:%M:%S")
        return f"[{data_formatada}] {self.tipo.value}: R$ {self.valor:.2f}"

In [ ]:
from abc import ABC, abstractmethod

"""classe abstrata não vira objeto, ela só é usada em subclasses"""
class Conta(ABC):
    def __init__(self, numero: int, titular: "Cliente", saldo_inicial: float = 0.0):

        self.numero: int = numero
        self.__saldo: float = saldo_inicial # Alterado para atributo privado
        self.titular: "Cliente" = titular
        self.transacoes: list["Transacao"] = (
            []
        )

        titular.adicionar_conta(self)
        """encapsulamento"""

    @property
    def saldo(self) -> float: # Propriedade pública para acessar o saldo
        return self.__saldo

    @saldo.setter
    def saldo(self, valor: float) -> None: # Setter para permitir a modificação do saldo
        if valor < 0:
            raise ValueError("Saldo não pode ser negativo.")
        self.__saldo = valor

    def consultar_saldo(self) -> float:
        return self.saldo # Usa a propriedade saldo

    def registrar_transacao(self, tipo: str, valor: float) -> None:
        if tipo.upper() == "DEPÓSITO" or tipo.upper() == "DEPOSITO":
            enum_tipo = TipoTransacao.DEPOSITO
        elif tipo.upper() == "SAQUE":
            enum_tipo = TipoTransacao.SAQUE
        else:
            enum_tipo = TipoTransacao.TRANSFERENCIA

        nova_transacao = Transacao(enum_tipo, valor)
        self.transacoes.append(nova_transacao)

    def depositar(self, valor: float) -> bool:
        if valor > 0:
            self.saldo += valor # Usa a propriedade saldo
            self.registrar_transacao("DEPÓSITO", valor)
            return True
        print("Valor de depósito inválido.")
        return False

    @abstractmethod #método abstrato
    def sacar(self, valor: float) -> bool:
        pass

    def transferir(self, destino: "Conta", valor: float) -> bool:
        if valor <= 0:
            print("Valor de transferência inválido.")
            return False

        if isinstance(destino, ContaPoupanca) and valor > 500.0:
            print("Erro: Transferências para Conta Poupança não podem ultrapassar R$ 500.00 por operação.")
            return False

        if self.sacar(valor):
            self.transacoes.pop()
            self.registrar_transacao("TRANSFERÊNCIA", valor)

            destino.saldo += valor # Usa a propriedade saldo
            destino.registrar_transacao("TRANSFERÊNCIA", valor)
            return True

        return False

    def exibir_extrato(self) -> None:
        print(f"\n=== Extrato da Conta Nº {self.numero} ===")
        if not self.transacoes:
            print("Nenhuma transação realizada.")
        else:
            for t in self.transacoes:
                print(t)
        print(f"Saldo Atual: R$ {self.consultar_saldo():.2f}")

In [ ]:
class ContaCorrente(Conta):#herança
#método instancia
    def sacar(self, valor: float) -> bool:
        taxa = valor * 0.10
        valor_total_debito = valor + taxa

        if valor > 0 and self.saldo >= valor_total_debito:
            self.saldo -= valor_total_debito
            self.registrar_transacao("SAQUE", valor)
            print(f"Taxa de 10% aplicada (R$ {taxa:.2f}). Total debitado: R$ {valor_total_debito:.2f}")
            return True

        print("Saldo insuficiente para cobrir o saque + taxa de 10%.")
        return False

In [ ]:
class ContaPoupanca(Conta):#herança
#método de instancia
    def sacar(self, valor: float) -> bool:
        if valor > 0 and self.saldo >= valor:
            self.saldo -= valor
            self.registrar_transacao("SAQUE", valor)
            return True
        print("Saldo insuficiente na Poupança.")
        return False

In [ ]:
class TipoTransacao(Enum):
    DEPOSITO = "DEPÓSITO"
    SAQUE = "SAQUE"
    TRANSFERENCIA = "TRANSFERÊNCIA"


class Transacao:
    """Representa uma operação bancária realizada na conta."""

    def __init__(self, tipo: TipoTransacao, valor: float):
        self.data: datetime = datetime.now()  # Atributo: data
        self.tipo: TipoTransacao = tipo  # Atributo: tipo (Enum)
        self.valor: float = valor  # Atributo: valor

    def __str__(self) -> str:
        data_formatada = self.data.strftime("%d/%m/%Y %H:%M:%S")
        return f"[{data_formatada}] {self.tipo.value} - R$ {self.valor:.2f}"

In [ ]:
def executar_sistema():
    nome = input("Digite seu nome: ")
    cpf = input("Digite seu CPF: ")
    email = input("Digite seu email: ")

    cliente = Cliente(nome, cpf, email)

    cc = ContaCorrente(numero=1001, titular=cliente, saldo_inicial=0.0)
    cp = ContaPoupanca(numero=2001, titular=cliente, saldo_inicial=0.0)
    cliente.adicionar_conta(cc)
    cliente.adicionar_conta(cp)

    while True:
        print(f"\n Caixa Eletronico - Olá, {cliente.nome} ")
        print("1 - Depositar")
        print("2 - Sacar")
        print("3 - Transferir")
        print("    1 - Conta Corrente -> Conta Poupança")
        print("    2 - Conta Poupança -> Conta Corrente")
        print("4 - Consultar Transações")
        print("5 - Consultar Saldo")
        print("0 - Sair")

        opcao = input("Escolha uma opção: ").strip()

        if opcao == "1":
            print("\nOnde deseja depositar?")
            print("1 - Conta Corrente")
            print("2 - Conta Poupança")
            tipo = input("Opção: ")
            try:
                valor = float(input("Digite o valor do depósito: R$ "))
                if tipo == "1":
                    conta = cc
                elif tipo == "2":
                    conta = cp
                else:
                    print("Opção de conta inválida.")
                    continue

                if conta.depositar(valor):
                    print("Depósito realizado com sucesso!")
            except ValueError:
                print("Valor inválido. Por favor, digite um número.")

        elif opcao == "2":
            print("\nDe onde deseja sacar?")
            print("1 - Conta Corrente (Atenção: Taxa de 10%)")
            print("2 - Conta Poupança")
            tipo = input("Opção: ")
            try:
                valor = float(input("Digite o valor do saque: R$ "))
                if tipo == "1":
                    conta = cc
                elif tipo == "2":
                    conta = cp
                else:
                    print("Opção de conta inválida.")
                    continue

                if conta.sacar(valor):
                    print("Saque realizado com sucesso!")
            except ValueError:
                print("Valor inválido. Por favor, digite um número.")

        elif opcao == "3":
            sub_opcao = input("Escolha a opção de transferência (1 ou 2): ").strip()
            try:
                valor = float(input("Digite o valor da transferência: R$ "))

                if sub_opcao == "1":
                    if cc.transferir(cp, valor):
                        print("Transferência realizada!")
                elif sub_opcao == "2":
                    if cp.transferir(cc, valor):
                        print("Transferência realizada!")
                else:
                    print("Opção de transferência inválida.")
            except ValueError:
                print("Valor inválido. Por favor, digite um número.")

        elif opcao == "4":
            print("\nQual extrato deseja consultar?")
            print("1 - Conta Corrente")
            print("2 - Conta Poupança")
            tipo = input("Opção: ")

            if tipo == "1":
                cc.exibir_extrato()
            elif tipo == "2":
                cp.exibir_extrato()
            else:
                print("Opção de conta inválida.")

        elif opcao == "5":
            cliente.listar_contas()

        elif opcao == "0":
            print("\nSistema encerrado. Obrigado por utilizar o Banco POO!")
            break
        else:
            print("Opção inválida! Tente novamente.")


if __name__ == "__main__":
    executar_sistema()
